# Variant calling with **GATK** — from BWA alignments to a hard-call VCF
### *Hypolimnas misippus* — continuation of `QC_Alignment_Hypolimnas_misippus_Colab.ipynb`

*Practical for the course "Genome-Wide Association Studies (GWAS) for Evolutionary Biology" (prstats.org)*

---

## Where are we in the workflow?

```
 FASTQ ─► QC_Alignment_Hypolimnas_misippus_Colab.ipynb ─► bams/<sample>.sorted.bam  (BWA-MEM, coordinate sorted)
                                                              │
                                                              ▼
                                               ► THIS NOTEBOOK (GATK best practices, germline short variants) ◄
                                                              │
                      MarkDuplicates ─► (indel realignment) ─► BQSR ─► HaplotypeCaller (GVCF)
                                       ─► joint genotyping ─► hard filtering ─► HypMis_GATK_hardcalls.vcf.gz
```

The input is the `bams/` folder produced by the QC/alignment notebook. The output is a **hard-call VCF**: for every variant and every individual one genotype (`0/0`, `0/1`, `1/1`, or `./.` when the call is not reliable enough).
This is the kind of file used by PLINK, GEMMA, `vcftools`, `bcftools`, PCA software and most population-genetics tools.

## Learning objectives

After this practical you should be able to

1. describe the **GATK Best Practices** workflow for germline short variants, and say *why* each step exists;
2. explain what **duplicates**, **local realignment**, **base quality score recalibration (BQSR)**, **GVCFs**, **joint genotyping**, and **variant filtering** do to the data;
3. adapt the workflow to a **non-model organism**, where there is no database of known variants and therefore **no VQSR and no standard BQSR** (bootstrapping and hard filters take their place);
4. judge the quality of a call set (Ti/Tv, missingness, depth, filter statistics);
5. produce a clean, documented hard-call VCF and know what each filter removed.

## Sections

| Part | Content |
|---|---|
| 0 | Background (the “lecture”): the logic of the GATK workflow |
| 1 | Install conda + software, mount Drive, parameters |
| 2 | Find the BAMs and the reference, sanity checks, extract the chromosome(s) to analyse |
| 3 | **MarkDuplicates** |
| 4 | **Indel realignment** (GATK 3.8 `IndelRealigner`; optional, and why GATK4 does not need it) |
| 5 | **Calling functions** — HaplotypeCaller → GenomicsDBImport → GenotypeGVCFs |
| 6 | **BQSR without known variants** (bootstrapping) |
| 7 | Final calling on the recalibrated BAMs |
| 8 | **Hard filtering** and the final hard-call VCF |
| 9 | Quality report, comparison, discussion, troubleshooting, exercises |

> **Runtime and size.** The full genome is too large for a free Colab session. By default we analyse **a 3 Mb window of one small chromosome (label 26, which carries the M locus of the colour-morph study)** for 20 individuals:
> about **1–2 hours** in total with 2 CPUs. Every expensive result (GVCFs, VCFs) is written to Drive and **re-used** if you re-run the notebook, so a Colab disconnection only costs you the cheap steps.
> To analyse more, change `REGION` / `ANALYSIS_LABEL` / `N_SAMPLES` in the parameter cell. The workflow is identical; only the clock time grows.

# Part 0 — Background: the GATK workflow in plain words

## 0.1 Why not just count alleles in a pileup?

A variant caller has to separate **real variation** from **errors**. The errors come from three places:

| Source of error | Example | How the workflow addresses it |
|---|---|---|
| **Library preparation** | PCR duplicates: one DNA molecule is sequenced several times, so a single sequencing error (or a single allele) is counted several times | **MarkDuplicates** |
| **Alignment** | Reads over an insertion/deletion are often aligned with mismatches instead of a gap, which creates *false SNPs* around indels, in different ways in different reads | **Indel realignment** (GATK3) / **local de novo assembly in HaplotypeCaller** (GATK4) |
| **Sequencer** | The base quality (Phred) scores reported by the machine are systematically over- or under-confident, depending on cycle, dinucleotide context, etc. | **BQSR** (base quality score recalibration) |
| **Statistics** | Low depth: one read with an alt allele can be an error or a heterozygote | genotype likelihoods, joint calling across individuals, **filtering** |

## 0.2 The steps

**1. MarkDuplicates (Picard).** Read pairs with identical 5′ start positions on both ends are flagged as likely duplicates (SAM flag `0x400`). Downstream tools ignore flagged reads. Typical duplicate rates in PCR-based libraries are 5–30%; for low-input insect libraries they can be higher.

**2. Indel realignment (GATK 3.x only: `RealignerTargetCreator` + `IndelRealigner`).**
* *RealignerTargetCreator* finds intervals where reads show signs of misalignment (indels in the CIGAR, clusters of mismatches).
* *IndelRealigner* builds candidate consensus sequences in these intervals, scores each read against reference and consensus, and **realigns** all reads to the best consensus. It works best **jointly across all samples**, because a consensus supported by several individuals is more reliable.

> **Important for teaching:** in **GATK4 this step was removed**. HaplotypeCaller (since GATK 3.x) already discards the original alignments in “active” regions, assembles the reads into a De Bruijn graph, finds candidate haplotypes, and re-aligns each read to every haplotype with the pair-HMM. This does what the realigner did, with better statistics, so the Broad Institute no longer recommends a separate realignment step before HaplotypeCaller.
> The realignment is still very informative to *see*, and it is still useful for callers that don't reassemble (e.g. samtools/bcftools, FreeBayes with some settings, VarScan, ANGSD!). This notebook runs it with GATK 3.8 in a separate conda environment so you can *see its effect*; it is **optional** (`RUN_INDEL_REALIGNMENT`) and the rest of the notebook works with or without it.

**3. BQSR (base quality score recalibration).** Models the empirical error rate as a function of the *reported* quality, read group, machine cycle and sequence context. It needs to know **which mismatches are real variants**; all other mismatches are counted as errors. Humans have databases (dbSNP, 1000G, Mills indels); a butterfly doesn't. The standard solution for non-model organisms is **bootstrapping**:

```
call variants with the raw qualities ─► keep only the most confident SNPs/indels (hard filters)
        ─► use them as "known sites" ─► BaseRecalibrator ─► ApplyBQSR ─► call again
        (optionally repeat until the recalibration tables stop changing)
```

BQSR is of limited value for very low coverage data (the recalibration needs ≳ 10⁷ bases per read group) and with few confident variants; the notebook checks for that.

**4. HaplotypeCaller in GVCF mode.** For each individual, HaplotypeCaller finds “active regions”, re-assembles reads, and computes the likelihood of each genotype with the pair-HMM. With `-ERC GVCF` it writes **a record for every position** (variant sites, plus blocks of reference-confidence summarised by their minimum GQ/DP). Why? So that *later*, when a new individual is added or a site becomes variable in another individual, one can still tell *“this individual was confidently homozygous reference there”* vs *“we had no data”*. A normal VCF cannot distinguish these. This is the “N+1 problem” solution.

**5. Joint genotyping (GenomicsDBImport → GenotypeGVCFs).** The GVCFs of all individuals are combined and genotypes are computed **jointly**: information is shared across individuals (the allele frequency acts as a prior; low-evidence alleles are only reported if they are supported somewhere). Joint calling is much better for low-coverage population data than calling each individual alone.

**6. Variant filtering.** VQSR (Variant Quality Score Recalibration) trains a Gaussian mixture model on *truth sets*; for non-model organisms there are none, so we use **hard filters** on annotations (QD, FS, SOR, MQ, MQRankSum, ReadPosRankSum). The standard thresholds from the GATK team are a *starting point*, not a rule: **look at the distributions** (we do, in Part 8) and adjust.

| Annotation | Meaning | Typical failure threshold (SNPs) |
|---|---|---|
| **QD** | QualByDepth: variant confidence divided by depth (normalises the fact that more reads → bigger QUAL) | `< 2` |
| **QUAL** | Phred-scaled confidence that a variant exists | `< 30` |
| **FS** | FisherStrand: strand bias (variant seen on only one strand = artefact) | `> 60` (indels `> 200`) |
| **SOR** | StrandOddsRatio: strand bias, better behaved at high depth | `> 3` |
| **MQ** | RMS mapping quality of reads at the site (repeats, paralogs have low MQ) | `< 40` |
| **MQRankSum** | alt vs. ref reads: are alt reads mapped worse? | `< −12.5` |
| **ReadPosRankSum** | alt vs. ref: are alt alleles near the read ends? | `< −8` (indels `< −20`) |

On top of these *site* filters we also apply **genotype** filters (`DP`, `GQ`) that set uncertain genotypes to missing, and then population-style filters (missingness, maximum depth). The last step is what makes the VCF a **hard-call** VCF.

## 0.3 Useful facts and warnings

* **Ploidy.** `-ploidy 2`: correct for autosomes. *Hypolimnas* females are **ZW**: on the Z chromosome females are effectively haploid, so a diploid caller will produce wrong heterozygote calls in females. This is why the previous notebook analysed autosomes only (`INCLUDE_Z=False`).
* **Heterozygosity prior.** GATK's default `--heterozygosity 0.001` is a human value. Insect populations typically have π ≈ 0.5–2%: we use **0.01** (parameter `HET_PRIOR`). Too low a prior makes the caller reluctant to call real variants at low depth.
* **Read groups.** GATK refuses BAMs without read groups, and BQSR needs the platform (`PL`) tag. The QC notebook wrote `ID`, `SM`, `PL`, `LB`; we verify it.
* **Reference.** GATK needs the reference FASTA **plus** `.fai` (samtools) and `.dict` (Picard). Contig names must be identical to the BAM headers; they are, because we align and call against the same file.
* **The “hard-call” caveat for low coverage.** GATK's genotype calls are only as good as the depth. At ≲ 5× per individual, many heterozygotes are called as homozygous. If your depth is low, ANGSD-style genotype *likelihoods* (see the previous notebooks) are statistically better; GATK's strength is accurate calls where depth is moderate to high.

## 0.4 Genotype calling output: what a VCF record looks like

```
#CHROM  POS    ID  REF ALT QUAL  FILTER INFO                         FORMAT       sample1      sample2
chr26   6731020 .  A   G   812.2 PASS   AC=4;AF=0.1;DP=250;QD=24.1;…  GT:AD:DP:GQ:PL  0/1:12,9:21:99:230,0,350  0/0:15,0:15:45:0,45,600
```

* `GT` genotype, `AD` allelic depths (ref,alt), `DP` depth, `GQ` genotype quality (Phred; 99 = max), `PL` Phred-scaled likelihoods of the three genotypes (0 = most likely).
* The *hard* call is derived from PL/GQ; filtering on `GQ` and `DP` decides which calls we keep.

# Part 1 — Setup: conda, software, Google Drive

We first install **condacolab**. This **restarts the Python kernel once** (Colab may show an ugly `[object CloseEvent]` message; this is normal). After the restart, **do not re-run this cell**: continue with the next cell.

In [ ]:
# 1.1  Install conda inside Colab (the kernel restarts automatically; ~30 s)
import os, sys, shutil
if shutil.which("mamba") or shutil.which("conda"):
    print("conda already available:", shutil.which("mamba") or shutil.which("conda"))
else:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
# 1.2  Mount Google Drive (a pop-up asks for permission)
from google.colab import drive
drive.mount("/content/drive")

## 1.3 Install the software

| Tool | Version/source | Used for |
|---|---|---|
| **GATK4** (includes Picard) | `bioconda::gatk4` | MarkDuplicates, BaseRecalibrator, ApplyBQSR, HaplotypeCaller, GenomicsDBImport, GenotypeGVCFs, SelectVariants, VariantFiltration |
| **GATK 3.8** + Java 8 | `bioconda::gatk` in a **separate environment** `gatk3` | `RealignerTargetCreator`, `IndelRealigner` (GATK4 does not contain them) |
| samtools, bcftools, htslib | bioconda | BAM/VCF handling, statistics |
| numpy, pandas, matplotlib, seaborn | conda-forge | tables and plots |

GATK4 needs Java ≥ 17 and GATK3 needs Java 8: this is why GATK3 lives in its own environment (`conda create -n gatk3`). The cell reports which tools were found; if GATK 3.8 cannot be installed the realignment step is skipped (the rest of the workflow does not depend on it).

In [ ]:
# 1.3  Install tools (takes a few minutes)
import subprocess, shutil, os, glob, re, time

def sh(cmd):
    return subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)

CONDA = shutil.which("mamba") or shutil.which("conda")
print("package manager:", CONDA)

if not shutil.which("gatk"):
    r = sh(f"{CONDA} install -y -q -c conda-forge -c bioconda gatk4 samtools bcftools htslib numpy pandas matplotlib seaborn scipy")
    if r.returncode: print(r.stderr[-1500:])
for t in ("gatk", "samtools", "bcftools", "bgzip", "tabix"):
    print(f"{t:9s}", shutil.which(t))
assert shutil.which("gatk") and shutil.which("samtools") and shutil.which("bcftools"), "installation failed – see the messages above"

# ---- GATK 3.8 (optional) in its own environment, with Java 8
ENV3 = None
base_prefix = sh(f"{CONDA} info --base").stdout.strip()
cand = f"{base_prefix}/envs/gatk3"
if not os.path.exists(cand + "/bin/java"):
    r = sh(f"{CONDA} create -y -q -n gatk3 -c conda-forge -c bioconda gatk")
    if r.returncode: print("GATK3 environment could not be created:\n", r.stderr[-800:])
if os.path.exists(cand + "/bin/java"):
    ENV3 = cand
GATK3_JAR = None
if ENV3:
    jars = glob.glob(ENV3 + "/**/GenomeAnalysisTK*.jar", recursive=True) + glob.glob("/content/drive/MyDrive/**/GenomeAnalysisTK*.jar", recursive=True)
    GATK3_JAR = jars[0] if jars else None
GATK3_WRAPPER = (ENV3 + "/bin/gatk3") if ENV3 and os.path.exists(ENV3 + "/bin/gatk3") else None
print("\nGATK 3.8 environment:", ENV3, "| jar:", GATK3_JAR, "| wrapper:", GATK3_WRAPPER)
print(sh("gatk --version 2>&1 | head -3").stdout)

> **If GATK 3.8 is not available** (no `jar`/`wrapper` printed above) you can download `GenomeAnalysisTK.jar` (3.8-1) yourself and put it anywhere in your Drive; the cell finds it by name. Otherwise the realignment step is skipped with a message.

## 1.4 Locate the project and set the parameters

Same logic as in the previous notebooks: the notebook searches your Drive for a folder called **`bams`** that contains `*.sorted.bam` files, and uses its **parent** as the project folder (where `phenotypes.tsv` and `reference_HypMis_v2/` live). Results are written to `gatk_HypMis/` in that project folder.

All analysis parameters are in this one cell.

In [ ]:
# 1.4  Parameters, paths
import os, re, json, gzip, glob, shutil, subprocess, io, time, math
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

DRIVE_ROOT = Path("/content/drive/MyDrive")
BAM_FOLDER_NAME = "bams"
BAM_DIR_OVERRIDE = None            # e.g. "/content/drive/MyDrive/GWAS_course/bams"
REF_FASTA_OVERRIDE = None          # e.g. "/content/drive/MyDrive/GWAS_course/reference_HypMis_v2/genome.fa"

# ================================================================ PARAMETERS
THREADS = os.cpu_count() or 2
FORCE = False                      # True = redo every step even if its output exists on disk

# ---- what to analyse
N_SAMPLES = 20                     # number of individuals (balanced between morphs); None = all BAMs
SEED = 42
CASE_MORPH = "m"                   # only used for labelling plots
ANALYSIS_LABEL = "26"              # chromosome label (from the FASTA header) to analyse; the M locus is on chromosome 26
CONTIGS_OVERRIDE = None            # e.g. ["JBBEEQ010000026.1"] or a list of several contigs; overrides ANALYSIS_LABEL
REGION = (5_000_000, 8_000_000)    # (start, end) bp window inside the contig used for calling; None = whole contig. Ignored if several contigs.

# ---- pre-processing
RUN_MARKDUP = True
RUN_INDEL_REALIGNMENT = True       # needs GATK 3.8 (see 1.3); skipped automatically if not available
BQSR_ROUNDS = 1                    # 0 = skip BQSR; 1 = bootstrapped BQSR (one round); more rounds are rarely worth it
BQSR_MIN_KNOWN_SNPS = 1000         # do not recalibrate if the bootstrap call set has fewer confident SNPs than this

# ---- HaplotypeCaller / GenotypeGVCFs
HET_PRIOR = 0.01                   # --heterozygosity (GATK default 0.001 is for humans)
PLOIDY = 2
MIN_BASEQ_HC = 10                  # HaplotypeCaller default
N_PAR = 2                          # samples processed in parallel (one Java process each)
MEM_GB = None                      # Java heap per process; None = chosen from the available RAM

# ---- site hard filters (GATK team recommendations; look at the distributions in Part 8 and adjust!)
SNP_FILTERS = {"QD2": "QD < 2.0", "QUAL30": "QUAL < 30.0", "SOR3": "SOR > 3.0", "FS60": "FS > 60.0",
               "MQ40": "MQ < 40.0", "MQRankSum-12.5": "MQRankSum < -12.5", "ReadPosRankSum-8": "ReadPosRankSum < -8.0"}
INDEL_FILTERS = {"QD2": "QD < 2.0", "QUAL30": "QUAL < 30.0", "FS200": "FS > 200.0", "ReadPosRankSum-20": "ReadPosRankSum < -20.0"}

# ---- genotype and population-level filters (what turns the VCF into a hard-call file)
MIN_GT_DP = 3                      # genotypes with depth below this are set to missing (./.)
MIN_GQ = 20                        # genotypes with GQ below this are set to missing
MAX_DP_SD = 3                      # remove sites whose total depth > mean + MAX_DP_SD * sd (paralogs/repeats); None = off
MAX_MISSING = 0.5                  # remove sites with more than this fraction of missing genotypes
MIN_CARRIERS = 1                   # keep sites where at least this many individuals carry an alt allele
FINAL_SNPS_ONLY = True             # final file: biallelic SNPs only (indels are kept in a separate "all variants" file)

# ================================================================ locate project
def find_on_drive(name, kind, maxdepth=6):
    res = subprocess.run(["find", str(DRIVE_ROOT), "-maxdepth", str(maxdepth), "-type", kind, "-name", name], capture_output=True, text=True)
    return [Path(p) for p in res.stdout.splitlines() if p.strip()]

if BAM_DIR_OVERRIDE:
    BAM_DIR = Path(BAM_DIR_OVERRIDE)
else:
    hits = [p for p in find_on_drive(BAM_FOLDER_NAME, "d") if any(p.glob("*.sorted.bam"))]
    assert hits, f"No folder called '{BAM_FOLDER_NAME}' with *.sorted.bam files found on Drive. Run the QC notebook first, or set BAM_DIR_OVERRIDE."
    hits.sort(key=lambda p: (not (p.parent / "phenotypes.tsv").exists(), str(p)))
    if len(hits) > 1:
        print("WARNING: several matching folders, using the first:"); [print("   ", h) for h in hits]
    BAM_DIR = hits[0]
PROJECT_DIR = BAM_DIR.parent

OUT = PROJECT_DIR / "gatk_HypMis"                       # persistent (Drive)
D = {k: OUT / k for k in ["gvcf", "vcf", "tables", "figures", "logs"]}
WORK = Path("/content/work_gatk")                       # local disk: fast, temporary
for k in ["bams", "tmp", "ref", "db"]: (WORK / k).mkdir(parents=True, exist_ok=True)
for d in D.values(): d.mkdir(parents=True, exist_ok=True)

# memory per Java process
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
if MEM_GB is None: MEM_GB = int(max(2, min(8, (ram_gb - 2.5) / N_PAR)))
print(f"Project folder : {PROJECT_DIR}\nBAM folder     : {BAM_DIR} ({len(list(BAM_DIR.glob('*.sorted.bam')))} BAM files)\nOutput folder  : {OUT}")
print(f"CPU threads {THREADS}, RAM {ram_gb:.1f} GB → {N_PAR} parallel jobs × {MEM_GB} GB Java heap")

In [ ]:
# 1.5  Helpers: run a shell command (with log), skip finished steps, run many commands in parallel
def run(cmd, log=None, check=True, show=10, quiet=False, env=None):
    if not quiet: print("$", " ".join(cmd.split())[:700])
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash", env=env)
    text = (r.stdout or "") + (r.stderr or "")
    if log is not None: Path(log).write_text(f"$ {cmd}\n\n{text}")
    if r.returncode != 0 and check:
        print(text[-3500:]); raise RuntimeError(f"Command failed (exit {r.returncode}); output above. Log: {log}")
    if show and text.strip() and not quiet: print("\n".join(text.strip().splitlines()[-show:]))
    if not quiet: print(f"[{time.time()-t0:.1f} s, exit {r.returncode}]")
    return r

def done(*paths):
    return (not FORCE) and all(Path(p).exists() and Path(p).stat().st_size > 0 for p in paths)

def run_many(jobs, n_par=None, label="job"):
    # jobs: list of (name, cmd, logfile). Runs up to n_par at once; raises with the log tail if one fails.
    n_par = n_par or N_PAR
    t0 = time.time(); results = {}
    def one(job):
        name, cmd, log = job
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
        Path(log).write_text(f"$ {cmd}\n\n{(r.stdout or '')}{(r.stderr or '')}")
        return name, r.returncode, (r.stdout or "") + (r.stderr or "")
    with ThreadPoolExecutor(max_workers=n_par) as ex:
        for i, (name, rc, text) in enumerate(ex.map(one, jobs), 1):
            results[name] = rc
            print(f"  [{i}/{len(jobs)}] {label} {name}: {'ok' if rc == 0 else 'FAILED (exit %d)' % rc}   ({(time.time()-t0)/60:.1f} min elapsed)")
            if rc != 0: print("   " + "\n   ".join(text.strip().splitlines()[-8:]))
    bad = [n for n, rc in results.items() if rc != 0]
    if bad: raise RuntimeError(f"{len(bad)} {label}(s) failed: {bad[:5]} – see the logs in {D['logs']}")
    return results

def nvar(vcf):
    return int(run(f"bcftools view -H {vcf} | wc -l", quiet=True).stdout.strip() or 0)

GATK_JAVA = f'--java-options "-Xmx{MEM_GB}g -XX:+UseParallelGC -XX:ParallelGCThreads=1 -Djava.io.tmpdir={WORK/"tmp"}"'
print("helpers ready")

# Part 2 — Inputs: samples, reference, BAM checks

## 2.1 The sample table

As in the previous notebook: `phenotypes.tsv` (written by the QC notebook) lists samples and morphs; if it is missing we rebuild the table from the BAM file names and `Hypolimnas_misippus_CAMID_morph.txt`.
To keep the practical fast we use `N_SAMPLES` individuals, **balanced between the two morphs** (m and i) and chosen reproducibly (`SEED`).

In [ ]:
# 2.1  Sample table
def locate_file(name):
    p = PROJECT_DIR / name
    if p.exists(): return p
    hits = find_on_drive(name, "f")
    return hits[0] if hits else None

pheno_path = PROJECT_DIR / "phenotypes.tsv"
if pheno_path.exists():
    pheno = pd.read_csv(pheno_path, sep="\t")[["sampleID", "morph"]]
else:
    mf = locate_file("Hypolimnas_misippus_CAMID_morph.txt")
    assert mf is not None, "neither phenotypes.tsv nor Hypolimnas_misippus_CAMID_morph.txt found in the project folder"
    morph = pd.read_csv(mf, sep="\t"); morph.columns = ["sampleID", "morph"]
    bams = sorted(BAM_DIR.glob("*.sorted.bam"))
    pheno = pd.DataFrame({"sampleID": [b.name.replace(".sorted.bam", "") for b in bams]}).merge(morph, on="sampleID")
pheno = pheno[pheno.sampleID.map(lambda s: (BAM_DIR / f"{s}.sorted.bam").exists())].reset_index(drop=True)
print(f"{len(pheno)} samples with a BAM file; morphs: {pheno.morph.value_counts().to_dict()}")

if N_SAMPLES and N_SAMPLES < len(pheno):
    rng = np.random.default_rng(SEED)
    per = N_SAMPLES // pheno.morph.nunique()
    parts = [g.sample(n=min(per, len(g)), random_state=int(rng.integers(1e9))) for _, g in pheno.groupby("morph")]
    pheno = pd.concat(parts).sort_values("sampleID").reset_index(drop=True)
pheno["bam"] = [str(BAM_DIR / f"{s}.sorted.bam") for s in pheno.sampleID]
SAMPLES = list(pheno.sampleID); N = len(SAMPLES)
assert pheno.sampleID.is_unique
print(f"using {N} samples; morphs: {pheno.morph.value_counts().to_dict()}")
display(pheno.head())

## 2.2 BAM integrity and read groups

Two things GATK is strict about:

1. The file must be valid (`samtools quickcheck`) and **coordinate sorted** with an index.
2. Every read needs a **read group** with `ID`, `SM` (sample), `PL` (platform) and `LB` (library). `SM` becomes the sample name in the VCF; `PL` is required by BQSR; `LB` is used by MarkDuplicates to decide which reads can be duplicates of each other.

In [ ]:
# 2.2  Integrity + read groups
r = subprocess.run(["samtools", "quickcheck", "-v"] + list(pheno.bam), capture_output=True, text=True)
bad = [l for l in r.stdout.splitlines() if l.strip()]
print("samtools quickcheck:", "all BAM files OK" if not bad else f"{len(bad)} PROBLEMATIC files, e.g. {bad[:3]}")
assert not bad
def rg_info(bam):
    h = subprocess.run(["samtools", "view", "-H", bam], capture_output=True, text=True).stdout
    so = re.search(r"^@HD.*?SO:(\w+)", h, flags=re.M)
    rgs = [dict(kv.split(":", 1) for kv in l.split("\t")[1:] if ":" in kv) for l in h.splitlines() if l.startswith("@RG")]
    return {"SO": so.group(1) if so else None, "n_RG": len(rgs), "RG": rgs[0] if rgs else {}}
info = [rg_info(b) for b in pheno.bam]
pheno["sorted"] = [i["SO"] for i in info]; pheno["n_RG"] = [i["n_RG"] for i in info]
for tag in ("ID", "SM", "PL", "LB"): pheno[f"RG_{tag}"] = [i["RG"].get(tag) for i in info]
display(pheno[["sampleID", "sorted", "n_RG", "RG_ID", "RG_SM", "RG_PL", "RG_LB"]].head())
assert (pheno["sorted"] == "coordinate").all(), "BAMs must be coordinate sorted"
assert (pheno.n_RG >= 1).all(), "some BAM files have no read group"
sm_bad = pheno[pheno.RG_SM != pheno.sampleID]
assert sm_bad.empty, "SM tag differs from the sample ID for: " + ", ".join(sm_bad.sampleID.head())
NEED_RG_FIX = bool(pheno[["RG_PL", "RG_LB"]].isna().any().any() or (pheno.n_RG > 1).any())
print("read groups complete (ID, SM, PL, LB)" if not NEED_RG_FIX else "⚠ PL/LB missing or several RGs → read groups will be rewritten while extracting (Part 2.5)")

## 2.3 The reference genome

GATK needs the **same FASTA** that was used for BWA, plus a `.fai` index (samtools) and a sequence dictionary `.dict` (Picard). We look for the FASTA in the project's `reference_HypMis_v2/` folder (as in the previous notebook), copy it to the local disk and build the two index files.

In [ ]:
# 2.3  Reference FASTA + index + dictionary
if REF_FASTA_OVERRIDE: src = Path(REF_FASTA_OVERRIDE)
else:
    cands = []
    for pat in ("*.fa", "*.fasta", "*.fna", "*.fa.gz", "*.fasta.gz", "*.fna.gz"):
        cands += sorted((PROJECT_DIR / "reference_HypMis_v2").glob(pat)) if (PROJECT_DIR / "reference_HypMis_v2").exists() else []
    if not cands:
        cands = [Path(p) for pat in ("*.fa", "*.fna", "*.fasta") for p in glob.glob(f"{PROJECT_DIR}/**/{pat}", recursive=True)]
    assert cands, "reference FASTA not found: set REF_FASTA_OVERRIDE"
    src = cands[0]
REF = WORK / "ref" / "reference.fa"
if not done(REF):
    if str(src).endswith(".gz"): run(f"gunzip -c {src} > {REF}", show=0)
    else: shutil.copy(src, REF)
if not done(str(REF) + ".fai"): run(f"samtools faidx {REF}", show=0)
REF_DICT = REF.with_suffix(".dict")
if not done(REF_DICT):
    REF_DICT.unlink(missing_ok=True)
    run(f"gatk CreateSequenceDictionary -R {REF} -O {REF_DICT}", log=D["logs"] / "dict.log", show=3)
fai = pd.read_csv(str(REF) + ".fai", sep="\t", header=None, names=["contig", "length", "offset", "bl", "bw"])[["contig", "length"]]
print("reference:", src, f"→ {REF}\n{len(fai)} sequences, {fai.length.sum()/1e6:.0f} Mb")

# the BAM header must have the same contigs and lengths
hdr = subprocess.run(["samtools", "view", "-H", pheno.bam.iloc[0]], capture_output=True, text=True).stdout
sq = {l.split("\t")[1][3:]: int(l.split("\t")[2][3:]) for l in hdr.splitlines() if l.startswith("@SQ")}
assert sq == dict(zip(fai.contig, fai.length)), "contigs in the BAM header differ from the FASTA: BAMs were aligned to another reference!"
print("BAM header and FASTA agree (same contig names and lengths)")

## 2.4 Which chromosome(s)?

The FASTA headers carry chromosome labels (`... chromosome: 26 ...`). We pick the contig with label `ANALYSIS_LABEL` (default `26`, which contains the M locus) and restrict calling to the window `REGION` inside it.
If labels cannot be parsed the notebook takes the shortest sequence longer than 5 Mb. You can always set `CONTIGS_OVERRIDE`.

In [ ]:
# 2.4  Choose the contig(s) and the calling intervals
heads = subprocess.run(f"grep '>' {REF}", shell=True, capture_output=True, text=True).stdout.splitlines()
desc = {h[1:].split(" ", 1)[0]: (h[1:].split(" ", 1) + [""])[1] for h in heads}
fai["label"] = fai.contig.map(desc).str.extract(r"(?:chromosome|scaffold)[:\s_]+([A-Za-z0-9]+)", expand=False)
if CONTIGS_OVERRIDE: CONTIGS = list(CONTIGS_OVERRIDE)
else:
    hit = fai.loc[fai.label == str(ANALYSIS_LABEL), "contig"]
    if len(hit): CONTIGS = [hit.iloc[0]]
    else:
        big = fai[fai.length > 5e6].sort_values("length")
        CONTIGS = [big.contig.iloc[0] if len(big) else fai.sort_values("length").contig.iloc[-1]]
        print("⚠ label not found, using", CONTIGS)
flen = dict(zip(fai.contig, fai.length))
if REGION and len(CONTIGS) == 1:
    s0, e0 = REGION; s0 = max(1, s0); e0 = min(e0, flen[CONTIGS[0]])
    INTERVALS = [f"{CONTIGS[0]}:{s0}-{e0}"]
else:
    INTERVALS = list(CONTIGS)
L_ARGS = " ".join(f"-L {x}" for x in INTERVALS)
print("contig(s):", CONTIGS, "| lengths (Mb):", [round(flen[c]/1e6, 2) for c in CONTIGS])
print("calling intervals:", INTERVALS)

## 2.5 Extract the chosen contig(s) to the local disk

To keep everything fast we copy only the **reads of the chosen contig(s)** from each BAM on Drive to the local disk (`/content/work_gatk/bams/`) and index them.
If the read groups were incomplete (see 2.2), they are rewritten here with `samtools addreplacerg`.

> *Caveat:* a read whose mate maps to *another* contig loses its mate in the extracted BAM. This affects a few percent of reads and does not matter for this practical; for a production run you would process complete BAMs.

In [ ]:
# 2.5  Extract contig reads → local BAMs
B0 = WORK / "bams" / "extract"; B0.mkdir(exist_ok=True)
jobs = []
for s, bam in zip(SAMPLES, pheno.bam):
    out = B0 / f"{s}.bam"
    if done(out, str(out) + ".bai"): continue
    if NEED_RG_FIX:
        cmd = (f"samtools view -@2 -b {bam} {' '.join(CONTIGS)} | samtools addreplacerg -@2 "
               f"-r '@RG\\tID:{s}\\tSM:{s}\\tPL:ILLUMINA\\tLB:lib1\\tPU:unit1' -m overwrite_all -O BAM -o {out} -")
    else:
        cmd = f"samtools view -@2 -b -o {out} {bam} {' '.join(CONTIGS)}"
    jobs.append((s, cmd + f" && samtools index {out}", D["logs"] / f"extract_{s}.log"))
if jobs: run_many(jobs, n_par=THREADS, label="extract")
cov = []
for s in SAMPLES:
    n = int(run(f"samtools view -c -F 0x904 {B0/(s+'.bam')}", quiet=True).stdout)
    cov.append(n)
pheno["reads_extracted"] = cov
print(f"extracted: median {int(np.median(cov)):,} primary mapped reads per sample (min {min(cov):,}, max {max(cov):,})")

## 2.6 How deep is the sequencing?

Depth decides what is reasonable later (genotype filters, BQSR, expected missingness). `samtools coverage` reports the mean depth over the chosen region.

In [ ]:
# 2.6  Depth per sample over the calling interval
def mean_depth(bam, interval):
    r = run(f"samtools coverage -r {interval} {bam}", quiet=True).stdout.strip().splitlines()[-1].split("\t")
    return float(r[6]), float(r[5])         # meandepth, coverage(%)
md_, cv_ = zip(*[mean_depth(B0 / f"{s}.bam", INTERVALS[0]) for s in SAMPLES])
pheno["mean_depth"], pheno["pct_covered"] = md_, cv_
print(f"mean depth in {INTERVALS[0]}: median {np.median(md_):.1f}×  (min {min(md_):.1f}, max {max(md_):.1f})")
fig, ax = plt.subplots(figsize=(8, 3))
colors = pheno.morph.map({"m": "#4c72b0", "i": "#dd8452"}).fillna("grey")
ax.bar(range(N), md_, color=colors); ax.set_xticks(range(N)); ax.set_xticklabels(SAMPLES, rotation=90, fontsize=7)
ax.set_ylabel("mean depth (×)"); ax.set_title("Sequencing depth in the analysed interval (blue = m, orange = i)")
plt.tight_layout(); plt.show()
if np.median(md_) < 5:
    print("⚠ median depth < 5×: expect many missing genotypes after the DP/GQ filters (consider MIN_GT_DP = 2 and MAX_MISSING = 0.7), "
          "and BQSR will have little data to learn from.")

# Part 3 — MarkDuplicates

**What it does.** `MarkDuplicates` (Picard, shipped inside GATK4) groups read pairs by their *unclipped 5′ coordinates* and orientation (and library). In each group it keeps the pair with the highest sum of base qualities and sets flag `0x400` on the others.

**Why it matters.** Duplicates arise during PCR in library preparation. A duplicate carries the same alleles *and the same errors* as the original, so they inflate the apparent evidence for a variant (or an error) and bias heterozygous genotype calls. HaplotypeCaller ignores reads with the duplicate flag.

**What to check.** The duplication rate (`PERCENT_DUPLICATION`) per sample: ≈ 5–20% is typical; very high values (> 50%) suggest low library complexity (too little DNA, too many PCR cycles) and **fewer useful reads than the raw depth suggests**.

Options used: `--CREATE_INDEX true` (so GATK3/GATK4 can read the BAM), `--VALIDATION_STRINGENCY SILENT` (the extracted BAMs contain reads whose mates lie elsewhere; we do not want warnings to stop the run).

In [ ]:
# 3.1  MarkDuplicates
B1 = WORK / "bams" / "markdup"; B1.mkdir(exist_ok=True)
if RUN_MARKDUP:
    jobs = []
    for s in SAMPLES:
        out = B1 / f"{s}.bam"
        if done(out, B1 / f"{s}.bai", D["tables"] / f"markdup_{s}.txt"): continue
        cmd = (f"gatk {GATK_JAVA} MarkDuplicates -I {B0/(s+'.bam')} -O {out} -M {D['tables']/('markdup_'+s+'.txt')} "
               f"--CREATE_INDEX true --VALIDATION_STRINGENCY SILENT --TMP_DIR {WORK/'tmp'}")
        jobs.append((s, cmd, D["logs"] / f"markdup_{s}.log"))
    if jobs: run_many(jobs, label="MarkDuplicates")
    # collect metrics
    rows = []
    for s in SAMPLES:
        lines = (D["tables"] / f"markdup_{s}.txt").read_text().splitlines()
        i = [k for k, l in enumerate(lines) if l.startswith("LIBRARY")][0]
        h = lines[i].split("\t"); v = lines[i + 1].split("\t")
        rows.append(dict(zip(h, v)) | {"sample": s})
    md_tab = pd.DataFrame(rows)
    md_tab["PERCENT_DUPLICATION"] = md_tab.PERCENT_DUPLICATION.astype(float) * 100
    pheno["dup_pct"] = md_tab.PERCENT_DUPLICATION.values
    print(f"duplication rate: median {md_tab.PERCENT_DUPLICATION.median():.1f}% (min {md_tab.PERCENT_DUPLICATION.min():.1f}, max {md_tab.PERCENT_DUPLICATION.max():.1f})")
    fig, ax = plt.subplots(figsize=(8, 3)); ax.bar(range(N), md_tab.PERCENT_DUPLICATION, color="#8172B2")
    ax.set_xticks(range(N)); ax.set_xticklabels(SAMPLES, rotation=90, fontsize=7); ax.set_ylabel("% duplicates"); ax.set_title("MarkDuplicates")
    plt.tight_layout(); plt.show()
    CUR = B1
else:
    CUR = B0; print("MarkDuplicates skipped (RUN_MARKDUP = False)")
print("current BAM set:", CUR)

# Part 4 — Indel realignment (GATK 3.8; optional)

## 4.1 The idea

Aligners score each read **independently**. A read that covers an insertion/deletion near its end often gets a gap-free alignment with a cluster of mismatches (cheaper than opening a gap); the same indel in another read is aligned with a gap. The mismatches look like a cluster of SNPs, **false variants flanking real indels**.

```
reference:   A C G T T T T T A C G        reference:   A C G T T T T T A C G
read 1:      A C G T T T T T A C G        read 1:      A C G T T T T T A C G
read 2 (aligned independently):           read 2 (after realignment, consensus with 1-T deletion):
             A C G T T T - T A C G                   A C G T T T - T A C G   ← same as read 3
read 3 (end of read, gap too costly):     read 3:
             A C G T T T G A C                        A C G T T T - T A C
                         ^ false mismatches                      ✓
```

**Two-step procedure (GATK 3):**

1. `RealignerTargetCreator` – scans all BAMs and writes intervals that *look* misaligned (known indels, indels in CIGAR, clusters of mismatches).
2. `IndelRealigner` – for each interval collects candidate consensus sequences and realigns every read to the consensus giving the lowest total mismatch score; reads are updated only if the improvement is significant (`LOD` threshold).

Run on **all individuals together** (`-I` for each BAM): a consensus supported by many individuals is more trustworthy and all individuals get the same representation of the indel.

## 4.2 Do we need it with HaplotypeCaller?

**No.** As explained in Part 0, GATK4 HaplotypeCaller re-assembles reads in each active region; the original alignment is only used to *find* the region. So the realigned BAM does not change HaplotypeCaller's calls much. We still run it (a) because it is part of the classical GATK workflow that you will find in many papers and pipelines; (b) to **see** what realignment does; (c) because realigned BAMs are useful input to callers that do **not** reassemble (bcftools, VarScan, ANGSD).

If GATK 3.8 is unavailable, the cell below explains and skips.

In [ ]:
# 4.3  RealignerTargetCreator + IndelRealigner (GATK 3.8, Java 8, separate conda environment)
B2 = WORK / "bams" / "realigned"; B2.mkdir(exist_ok=True)
TARGETS = D["tables"] / "realign_targets.intervals"

def gatk3_cmd(tool_args, mem=MEM_GB * N_PAR):
    # Build the command line for GATK 3.8: wrapper if present, else java -jar. PATH puts the Java 8 of the gatk3 env first.
    env = f"PATH={ENV3}/bin:$PATH JAVA_HOME={ENV3} "
    if GATK3_WRAPPER:  return f"{env} JAVA_TOOL_OPTIONS=-Xmx{mem}g {GATK3_WRAPPER} {tool_args}"
    return f"{env} {ENV3}/bin/java -Xmx{mem}g -Djava.io.tmpdir={WORK/'tmp'} -jar {GATK3_JAR} {tool_args}"

if RUN_INDEL_REALIGNMENT and ENV3 and (GATK3_WRAPPER or GATK3_JAR):
    ins = " ".join(f"-I {CUR/(s+'.bam')}" for s in SAMPLES)
    if not done(TARGETS):
        run(gatk3_cmd(f"-T RealignerTargetCreator -R {REF} {ins} {L_ARGS} -nt {THREADS} -o {TARGETS}"), log=D["logs"] / "realigner_targets.log", show=6)
    n_targets = sum(1 for _ in open(TARGETS))
    print(f"{n_targets:,} target intervals to realign")
    mapfile = WORK / "realign_map.txt"
    mapfile.write_text("".join(f"{CUR/(s+'.bam')}\t{B2/(s+'.bam')}\n" for s in SAMPLES))
    if not all(done(B2 / f"{s}.bam") for s in SAMPLES):
        run(gatk3_cmd(f"-T IndelRealigner -R {REF} {ins} -targetIntervals {TARGETS} {L_ARGS} -nWayOut {mapfile}"),
            log=D["logs"] / "indelrealigner.log", show=6)
    for s in SAMPLES:                                              # index (IndelRealigner may or may not write .bai)
        if not (done(B2 / f"{s}.bai") or done(str(B2 / f"{s}.bam") + ".bai")):
            run(f"samtools index {B2/(s+'.bam')}", quiet=True)
    REALIGNED = True
else:
    REALIGNED = False
    why = ("RUN_INDEL_REALIGNMENT = False" if not RUN_INDEL_REALIGNMENT else "GATK 3.8 is not available (see Part 1.3)")
    print(f"Indel realignment skipped ({why}). This is fine: GATK4 HaplotypeCaller re-assembles reads locally.")

### 4.4 What did the realignment change?

A simple, honest measure: count how many **reads carry an indel in their CIGAR** (letters `I`/`D`) and the number of **mismatches per read** (the `NM` tag) before and after, inside the calling interval. After realignment, reads over an indel agree on the gap (more reads with `I`/`D`) and the number of mismatches (`NM`) decreases a little. The effect is expected to be modest.

In [ ]:
# 4.4  Before / after statistics (first 6 samples, calling interval)
def indel_stats(bam, interval, maxreads=300000):
    p = subprocess.Popen(f"samtools view -F 0x904 {bam} {interval} | head -n {maxreads} | cut -f6,12-", shell=True, stdout=subprocess.PIPE, text=True)
    n = ind = nm_sum = 0
    for line in p.stdout:
        f = line.rstrip("\n").split("\t"); n += 1
        if re.search(r"[ID]", f[0]): ind += 1
        m = re.search(r"NM:i:(\d+)", "\t".join(f[1:]))
        nm_sum += int(m.group(1)) if m else 0
    p.stdout.close(); p.wait()
    return n, ind / max(n, 1), nm_sum / max(n, 1)

if REALIGNED:
    rows = []
    for s in SAMPLES[:6]:
        a = indel_stats(CUR / f"{s}.bam", INTERVALS[0]); b = indel_stats(B2 / f"{s}.bam", INTERVALS[0])
        rows.append({"sample": s, "reads": a[0], "% reads with I/D before": 100 * a[1], "after": 100 * b[1], "mean NM before": a[2], "mean NM after": b[2]})
    display(pd.DataFrame(rows).round(3))
    CUR = B2
else:
    print("(no realignment performed)")
print("current BAM set:", CUR)

# Part 5 — The calling functions: HaplotypeCaller → joint genotyping → hard filters

We need the calling steps **twice**: once for the *bootstrap* of BQSR (Part 6) and once for the final call set (Part 7). So we first define them as functions and explain each command.

## 5.1 HaplotypeCaller in GVCF mode

```
gatk HaplotypeCaller -R reference.fa -I sample.bam -O sample.g.vcf.gz -ERC GVCF -L chr:start-end \
     --heterozygosity 0.01 -ploidy 2
```

What happens inside, per individual:

1. **Active regions.** The genome is scanned for places with evidence of variation (mismatches, soft clips, indels). Everything else is declared “reference confidence” cheaply.
2. **Local re-assembly.** In an active region the original alignments are thrown away; reads are assembled into a De Bruijn graph; paths through the graph are candidate **haplotypes**.
3. **Pair-HMM.** Each read is scored against each candidate haplotype → likelihood P(read | haplotype), including base qualities and indel error models.
4. **Genotype likelihoods.** Read likelihoods are combined into likelihoods of each diploid genotype (`PL` field); with a prior (`--heterozygosity`) the best genotype and its quality (`GQ`) are reported.
5. **GVCF output** (`-ERC GVCF`): besides variant records there are `<NON_REF>` blocks summarising how confidently the individual is homozygous-reference (min `GQ`, min `DP` within the block).

The `-L` interval limits the work to our region. One Java process per individual; `N_PAR` of them run at the same time.

In [ ]:
# 5.1  HaplotypeCaller (GVCF mode) for all samples — resumable; GVCFs are cached on Drive
def call_gvcfs(bam_dir, tag):
    gdir = D["gvcf"] / tag; gdir.mkdir(exist_ok=True)
    ldir = WORK / "gvcf" / tag; ldir.mkdir(parents=True, exist_ok=True)
    jobs = []
    for s in SAMPLES:
        drv = gdir / f"{s}.g.vcf.gz"
        if done(drv, str(drv) + ".tbi"): continue
        loc = ldir / f"{s}.g.vcf.gz"
        cmd = (f"gatk {GATK_JAVA} HaplotypeCaller -R {REF} -I {bam_dir/(s+'.bam')} -O {loc} -ERC GVCF {L_ARGS} "
               f"--heterozygosity {HET_PRIOR} -ploidy {PLOIDY} --min-base-quality-score {MIN_BASEQ_HC} "
               f"--native-pair-hmm-threads 1 --tmp-dir {WORK/'tmp'} && cp {loc} {drv} && cp {loc}.tbi {drv}.tbi")
        jobs.append((s, cmd, D["logs"] / f"hc_{tag}_{s}.log"))
    if jobs:
        print(f"HaplotypeCaller [{tag}]: {len(jobs)} samples to call ({len(SAMPLES)-len(jobs)} cached on Drive), {N_PAR} in parallel")
        t0 = time.time(); run_many(jobs, label="HaplotypeCaller"); print(f"done in {(time.time()-t0)/60:.1f} min")
    else:
        print(f"HaplotypeCaller [{tag}]: all {len(SAMPLES)} GVCFs already on Drive")
    return gdir
print("call_gvcfs defined")

## 5.2 Joint genotyping: GenomicsDBImport → GenotypeGVCFs

* **GenomicsDBImport** loads all GVCFs into a columnar database (`GenomicsDB`) that can be queried by interval. This is much faster than merging GVCF files for large cohorts. (For a handful of samples `CombineGVCFs` is equivalent; the code falls back to it if GenomicsDBImport fails.)
* **GenotypeGVCFs** reads the database and calls **genotypes for all individuals at every site where at least one has evidence of a variant**. Because the allele frequency over all individuals is used as a prior, an alt allele supported by 1 read in 1 individual is only called if the whole picture makes it plausible.
  Output records with `QUAL < 30` (Phred) are not emitted (`--standard-min-confidence-threshold-for-calling 30`, the default).
* `--heterozygosity` should match what was used in HaplotypeCaller.
* GenotypeGVCFs adds the annotations we use for filtering: `QD, FS, SOR, MQ, MQRankSum, ReadPosRankSum, DP, AC, AF, AN`.

In [ ]:
# 5.2  Joint genotyping → VCF (cached on Drive)
def joint_genotype(gdir, tag, vcf_out):
    vcf_out = Path(vcf_out)
    if done(vcf_out, str(vcf_out) + ".tbi"):
        print(f"joint calls [{tag}] already on Drive:", vcf_out.name); return vcf_out
    ldir = WORK / "gvcf" / f"{tag}_joint"; ldir.mkdir(parents=True, exist_ok=True)
    for s in SAMPLES:                                              # GVCFs → local disk (GenomicsDB is much faster on local disk)
        for ext in ("", ".tbi"):
            if not (ldir / f"{s}.g.vcf.gz{ext}").exists(): shutil.copy(gdir / f"{s}.g.vcf.gz{ext}", ldir / f"{s}.g.vcf.gz{ext}")
    smap = ldir / "sample_map.txt"
    smap.write_text("".join(f"{s}\t{ldir/(s+'.g.vcf.gz')}\n" for s in SAMPLES))
    big = f'--java-options "-Xmx{MEM_GB * N_PAR}g -XX:+UseParallelGC -XX:ParallelGCThreads=2 -Djava.io.tmpdir={WORK/"tmp"}"'
    parts = []
    for k, iv in enumerate(INTERVALS):
        part = WORK / "tmp" / f"{tag}_part{k}.vcf.gz"; db = WORK / "db" / f"{tag}_{k}"
        shutil.rmtree(db, ignore_errors=True)
        r = run(f"gatk {big} GenomicsDBImport --genomicsdb-workspace-path {db} -L {iv} --sample-name-map {smap} "
                f"--reader-threads {min(THREADS, 4)} --batch-size 50 --tmp-dir {WORK/'tmp'}", log=D["logs"] / f"gdb_{tag}_{k}.log", check=False, show=3)
        if r.returncode == 0:
            src = f"gendb://{db}"
        else:
            print("GenomicsDBImport failed → falling back to CombineGVCFs (see the log)")
            comb = WORK / "tmp" / f"{tag}_{k}.combined.g.vcf.gz"
            vs = " ".join(f"-V {ldir/(s+'.g.vcf.gz')}" for s in SAMPLES)
            run(f"gatk {big} CombineGVCFs -R {REF} {vs} -L {iv} -O {comb}", log=D["logs"] / f"combine_{tag}_{k}.log", show=3)
            src = str(comb)
        run(f"gatk {big} GenotypeGVCFs -R {REF} -V {src} -O {part} -L {iv} --heterozygosity {HET_PRIOR} -ploidy {PLOIDY} --tmp-dir {WORK/'tmp'}",
            log=D["logs"] / f"genotype_{tag}_{k}.log", show=3)
        parts.append(part)
    merged = WORK / "tmp" / f"{tag}_joint.vcf.gz"
    if len(parts) == 1: shutil.copy(parts[0], merged); shutil.copy(str(parts[0]) + ".tbi", str(merged) + ".tbi")
    else:
        run(f"bcftools concat -a -Oz -o {merged} {' '.join(map(str, parts))} && bcftools index -t -f {merged}", show=2)
    shutil.copy(merged, vcf_out); shutil.copy(str(merged) + ".tbi", str(vcf_out) + ".tbi")
    return vcf_out
print("joint_genotype defined")

## 5.3 Hard filtering

GATK's recommended replacement for VQSR when no truth set exists. It is done **separately for SNPs and indels**, because the annotations behave differently (e.g. indels have more strand bias):

1. `SelectVariants --select-type-to-include SNP` / `INDEL` (+ `MIXED`, `MNP`) → two files;
2. `VariantFiltration --filter-expression "QD < 2.0" --filter-name QD2 ...` → adds a **FILTER** value to failing records (nothing is deleted yet; `PASS` = passed everything);
3. `MergeVcfs` joins them again.

A record with annotation values that are *missing* (e.g. `MQRankSum` is absent when there are no alt reads to compare) is **not** failed by that expression.

The result still has all variants, labelled `PASS` or with the names of the failed filters, so you can see exactly what each filter removed.

In [ ]:
# 5.3  Hard filter function (SNPs and indels separately)
def filter_args(filters):
    return " ".join(f'--filter-expression "{e}" --filter-name "{n}"' for n, e in filters.items())

def hard_filter(vcf, tag, snp_filters=None, indel_filters=None, out_dir=None):
    snp_filters = SNP_FILTERS if snp_filters is None else snp_filters
    indel_filters = INDEL_FILTERS if indel_filters is None else indel_filters
    out = Path(out_dir or D["vcf"]) / f"HypMis_{tag}_sitefiltered.vcf.gz"
    t = WORK / "tmp"
    run(f"gatk {GATK_JAVA} SelectVariants -R {REF} -V {vcf} --select-type-to-include SNP -O {t}/{tag}_snp.vcf.gz", log=D["logs"] / f"sel_snp_{tag}.log", show=0, quiet=True)
    run(f"gatk {GATK_JAVA} SelectVariants -R {REF} -V {vcf} --select-type-to-include INDEL --select-type-to-include MIXED --select-type-to-include MNP "
        f"-O {t}/{tag}_ind.vcf.gz", log=D["logs"] / f"sel_ind_{tag}.log", show=0, quiet=True)
    run(f"gatk {GATK_JAVA} VariantFiltration -R {REF} -V {t}/{tag}_snp.vcf.gz {filter_args(snp_filters)} -O {t}/{tag}_snp_f.vcf.gz", log=D["logs"] / f"vf_snp_{tag}.log", show=0, quiet=True)
    run(f"gatk {GATK_JAVA} VariantFiltration -R {REF} -V {t}/{tag}_ind.vcf.gz {filter_args(indel_filters)} -O {t}/{tag}_ind_f.vcf.gz", log=D["logs"] / f"vf_ind_{tag}.log", show=0, quiet=True)
    loc = t / f"{tag}_merged.vcf.gz"
    run(f"gatk {GATK_JAVA} MergeVcfs -I {t}/{tag}_snp_f.vcf.gz -I {t}/{tag}_ind_f.vcf.gz -O {loc}", log=D["logs"] / f"merge_{tag}.log", show=0, quiet=True)
    shutil.copy(loc, out); shutil.copy(str(loc) + ".tbi", str(out) + ".tbi")
    return out

def filter_counts(vcf):
    r = run(f"bcftools query -f '%FILTER\\t%TYPE\\n' {vcf}", quiet=True).stdout.splitlines()
    c = pd.Series(r).str.split("\t", expand=True); c.columns = ["FILTER", "TYPE"]
    c["TYPE"] = np.where(c.TYPE == "snp", "SNP", "indel/other")
    return c
print("hard_filter defined")

# Part 6 — BQSR without known variants (bootstrapping)

## 6.1 What BQSR does

The Phred quality scores reported by the sequencer are only *predictions* of the probability that a base is wrong. BQSR **measures** the true error rate:

1. For every base in the BAM, decide whether it **mismatches the reference**; if the position is *not* a known variant, the mismatch is counted as a sequencing error.
2. Tabulate observations and errors by **read group**, **reported quality**, **machine cycle** (position in the read) and **dinucleotide context**.
3. Fit a model: `empirical quality = reported quality + Δ(read group) + Δ(quality) + Δ(cycle) + Δ(context)`.
4. **ApplyBQSR** rewrites the base qualities (`QUAL` line in the BAM) using the model. HaplotypeCaller then uses better-calibrated quality scores in its likelihoods.

The critical input is the *known sites* VCF: real variants must **not** be counted as errors. If we left them out, every polymorphic site would look like a run of errors and the qualities would be pushed down.

## 6.2 The bootstrap

```
round 1:  HaplotypeCaller(raw BAMs) → joint genotyping → hard filters → PASS variants = "known sites"
          BaseRecalibrator(known sites) → ApplyBQSR → recalibrated BAMs → (Part 7) HaplotypeCaller again
```

In the bootstrap set it is better to be **conservative**: a few thousand reliable variants are enough, and *missing* some real variants only costs a little (they would be treated as errors, but at a negligible rate); including false positives has no effect either, as errors are rare. We therefore keep only `PASS` variants with `QUAL ≥ 50`.

> **Is it worth it?** For low-depth data BQSR often changes little. Use the plots in 6.4 to decide: if empirical and reported qualities already agree (points on the diagonal), the machine was well calibrated, and recalibration is not needed. You can set `BQSR_ROUNDS = 0` to skip it.

In [ ]:
# 6.1  Bootstrap: first-pass calls on the (realigned) dedup BAMs → known sites
BQSR_APPLIED = False
VCF1 = D["vcf"] / "HypMis_round1_raw.vcf.gz"
KNOWN = D["vcf"] / "known_sites_bootstrap.vcf.gz"
if BQSR_ROUNDS >= 1:
    g1 = call_gvcfs(CUR, "round1")
    joint_genotype(g1, "round1", VCF1)
    n_raw = nvar(VCF1)
    print(f"round-1 raw call set: {n_raw:,} variants")
    if not done(KNOWN, str(KNOWN) + ".tbi"):
        f1 = hard_filter(VCF1, "round1")
        run(f"bcftools view -f PASS -i 'QUAL>=50' -Oz -o {KNOWN} {f1} && bcftools index -t -f {KNOWN}", show=2)
    n_known_snp = int(run(f"bcftools view -H -v snps {KNOWN} | wc -l", quiet=True).stdout)
    n_known_ind = int(run(f"bcftools view -H -v indels {KNOWN} | wc -l", quiet=True).stdout)
    print(f"known sites for BQSR: {n_known_snp:,} SNPs, {n_known_ind:,} indels")
    if n_known_snp < BQSR_MIN_KNOWN_SNPS:
        print(f"⚠ fewer than {BQSR_MIN_KNOWN_SNPS} confident SNPs: not enough to learn from → BQSR skipped. (Low depth and few variants make recalibration unreliable.)")
else:
    print("BQSR_ROUNDS = 0: BQSR skipped")

## 6.3 BaseRecalibrator and ApplyBQSR

* `BaseRecalibrator` builds the recalibration table per sample (`--known-sites` = our bootstrap VCF; `-L` restricts the work to the calling intervals).
* `ApplyBQSR` writes new BAMs with recalibrated qualities (original ones are kept in the `OQ` tag only if `--use-original-qualities` was used when *building* the table; we don't).

In [ ]:
# 6.3  BaseRecalibrator + ApplyBQSR
B3 = WORK / "bams" / "recal"; B3.mkdir(exist_ok=True)
if BQSR_ROUNDS >= 1 and n_known_snp >= BQSR_MIN_KNOWN_SNPS:
    jobs = []
    for s in SAMPLES:
        tab = D["tables"] / f"recal_{s}.table"
        if done(tab): continue
        jobs.append((s, f"gatk {GATK_JAVA} BaseRecalibrator -R {REF} -I {CUR/(s+'.bam')} --known-sites {KNOWN} {L_ARGS} -O {tab} --tmp-dir {WORK/'tmp'}",
                     D["logs"] / f"bqsr_table_{s}.log"))
    if jobs: run_many(jobs, label="BaseRecalibrator")
    jobs = []
    for s in SAMPLES:
        out = B3 / f"{s}.bam"
        if done(out, B3 / f"{s}.bai"): continue
        jobs.append((s, f"gatk {GATK_JAVA} ApplyBQSR -R {REF} -I {CUR/(s+'.bam')} -bqsr {D['tables']/('recal_'+s+'.table')} -O {out} --tmp-dir {WORK/'tmp'}",
                     D["logs"] / f"bqsr_apply_{s}.log"))
    if jobs: run_many(jobs, label="ApplyBQSR")
    BQSR_APPLIED = True
print("BQSR applied:", BQSR_APPLIED)

## 6.4 Did recalibration matter? — reported vs. empirical quality

The recalibration table contains, for every reported quality `Q`, the number of observed bases and the number of mismatches that were counted as errors. The **empirical** quality is `−10·log10(errors/observations)`. A well-calibrated sequencer lies on the diagonal. Points *below* the diagonal mean the machine was over-confident (real error rate higher than reported); *above* mean under-confident.

In [ ]:
# 6.4  Parse GATK report tables and plot reported vs empirical quality
def read_gatk_table(path, name):
    lines = Path(path).read_text().splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(f"#:GATKTable:{name}:") or l.startswith(f"#:GATKTable:{name} "))
    hdr = lines[i + 1].split(); rows = []
    for l in lines[i + 2:]:
        if not l.strip() or l.startswith("#"): break
        rows.append(l.split())
    return pd.DataFrame(rows, columns=hdr)

if BQSR_APPLIED:
    qt = pd.concat([read_gatk_table(D["tables"] / f"recal_{s}.table", "RecalTable1") for s in SAMPLES])
    qt = qt[qt.EventType == "M"].copy()
    for c in ("QualityScore", "Observations", "Errors"): qt[c] = qt[c].astype(float)
    agg = qt.groupby("QualityScore")[["Observations", "Errors"]].sum().reset_index()
    agg["empirical"] = -10 * np.log10((agg.Errors + 1) / (agg.Observations + 2))
    wmean_shift = np.average(agg.empirical - agg.QualityScore, weights=agg.Observations)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].scatter(agg.QualityScore, agg.empirical, s=10 + 150 * agg.Observations / agg.Observations.max(), color="#4C72B0", alpha=.7)
    ax[0].plot([0, 45], [0, 45], "k--", lw=.8); ax[0].set_xlabel("reported base quality"); ax[0].set_ylabel("empirical quality"); ax[0].set_title("Calibration of base qualities (size = # bases)")
    ax[1].bar(agg.QualityScore, agg.Observations, color="grey"); ax[1].set_yscale("log"); ax[1].set_xlabel("reported base quality"); ax[1].set_ylabel("bases observed"); ax[1].set_title("Where the data are")
    plt.tight_layout(); plt.savefig(D["figures"] / "bqsr_calibration.png", dpi=150); plt.show()
    print(f"observation-weighted mean (empirical − reported) = {wmean_shift:+.2f} Phred; total bases used: {agg.Observations.sum():.2e}")
    print("negative = the sequencer was over-confident; recalibration lowers the qualities accordingly" if wmean_shift < 0 else "positive = the sequencer was under-confident")
else:
    print("(BQSR not applied: nothing to plot)")

# Part 7 — Final calling on the recalibrated BAMs

Now the same two functions are used on the BAMs that have recalibrated base qualities (or, if BQSR was skipped, on the pre-processed BAMs). The final call set is called **`final`**.

In [ ]:
# 7.1  Final HaplotypeCaller + joint genotyping
FINAL_BAMS = B3 if BQSR_APPLIED else CUR
VCF_RAW = D["vcf"] / "HypMis_final_raw.vcf.gz"
if (not BQSR_APPLIED) and BQSR_ROUNDS >= 1 and done(VCF1):
    print("BQSR was not applied → the round-1 call set is already the final one (same BAMs); re-using it")
    if not done(VCF_RAW):
        shutil.copy(VCF1, VCF_RAW); shutil.copy(str(VCF1) + ".tbi", str(VCF_RAW) + ".tbi")
else:
    gF = call_gvcfs(FINAL_BAMS, "final")
    joint_genotype(gF, "final", VCF_RAW)
n_final_raw = nvar(VCF_RAW)
print(f"final raw call set: {n_final_raw:,} variants in {N} individuals   ({VCF_RAW.name})")
if BQSR_APPLIED:
    print(f"(round 1 had {nvar(VCF1):,} variants: the difference comes from the recalibrated base qualities)")

# Part 8 — Variant filtering and the hard-call VCF

The raw joint call set (`HypMis_final_raw.vcf.gz`) contains every site where *some* individual has evidence for an alternative allele — including many false positives (alignment artefacts in repeats, strand-biased calls, …). We now filter in **four layers**; each one removes a different kind of problem:

| Layer | Level | Tool | Removes |
|---|---|---|---|
| 1. Site hard filters | site | `VariantFiltration` | low-confidence calls, strand bias, poor mapping |
| 2. Genotype filters | individual genotype | `VariantFiltration` + `SelectVariants` | genotypes based on too few reads or with low GQ → set to **missing** |
| 3. Depth filter | site | `bcftools` | sites with abnormally high total depth (collapsed repeats, paralogs) |
| 4. Missingness / carriers / type | site | `bcftools` | sites genotyped in too few individuals, monomorphic after filtering, indels or multiallelic sites |

## 8.1 First look at the annotations

Before applying the GATK default thresholds, **look at the distributions**. We export the annotations with `VariantsToTable` and plot SNPs and indels separately, with the filter threshold as a dashed line. Good thresholds cut off a *tail*, not the main mode of the distribution.

In [ ]:
# 8.1  Annotation table + distributions
ANN = D["tables"] / "final_raw_annotations.tsv"
if not done(ANN):
    run(f"gatk {GATK_JAVA} VariantsToTable -V {VCF_RAW} -F CHROM -F POS -F TYPE -F QUAL -F QD -F FS -F SOR -F MQ -F MQRankSum -F ReadPosRankSum -F DP -F AC -F AN "
        f"-O {ANN}", log=D["logs"] / "variantstotable.log", show=2)
ann = pd.read_csv(ANN, sep="\t", na_values=["NA"])
ann["class"] = np.where(ann.TYPE == "SNP", "SNP", "indel/other")
print(f"{len(ann):,} variants:", ann["class"].value_counts().to_dict())

def parse_expr(e):
    m = re.match(r"\s*(\w+)\s*([<>])\s*(-?[\d.]+)", e); return m.group(1), m.group(2), float(m.group(3))
thr = {"SNP": {parse_expr(e)[0]: parse_expr(e)[2] for e in SNP_FILTERS.values()},
       "indel/other": {parse_expr(e)[0]: parse_expr(e)[2] for e in INDEL_FILTERS.values()}}
cols = ["QD", "QUAL", "FS", "SOR", "MQ", "MQRankSum", "ReadPosRankSum", "DP"]
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, c in zip(axes.ravel(), cols):
    for cl, colr in (("SNP", "#4C72B0"), ("indel/other", "#DD8452")):
        v = ann.loc[ann["class"] == cl, c].dropna()
        if len(v) == 0: continue
        lo, hi = v.quantile(.005), v.quantile(.995)
        ax.hist(v.clip(lo, hi), bins=60, alpha=.55, color=colr, density=True, label=f"{cl} (n={len(v):,})")
        if c in thr[cl]: ax.axvline(thr[cl][c], color=colr, ls="--")
    ax.set_title(c); ax.legend(fontsize=7)
plt.suptitle("Annotation distributions of the raw call set (dashed = filter thresholds)")
plt.tight_layout(); plt.savefig(D["figures"] / "annotation_distributions.png", dpi=150); plt.show()

**How to read these panels**

* **QD** (quality by depth) should have a main peak at 15–35 for good variants and a tail towards 0 for artefacts. A threshold of 2 removes the tail only.
* **FS / SOR** (strand bias): most sites are near 0; the long tail to the right are strand-biased artefacts.
* **MQ**: real unique-sequence variants have MQ ≈ 60 (BWA's maximum); a left tail marks multi-mapping reads (repeats).
* **MQRankSum / ReadPosRankSum**: roughly symmetric around 0; only the very negative tail is suspect. (Missing for sites without alt reads to compare.)
* **DP** (total depth over all individuals): a main mode plus a long right tail of **over-covered sites** = collapsed repeats/paralogs that look like “variants” because reads from several genomic copies pile up at one place. We use a depth filter (layer 3) for that.

Each variant failing a filter is **flagged**, not deleted; the table below shows how many records each filter flags (a record can fail several filters).

In [ ]:
# 8.2  What would each hard filter remove?
rows = []
for cl, flt in (("SNP", SNP_FILTERS), ("indel/other", INDEL_FILTERS)):
    sub = ann[ann["class"] == cl]
    tot_fail = pd.Series(False, index=sub.index)
    for name, e in flt.items():
        c, op, v = parse_expr(e)
        fail = (sub[c] < v) if op == "<" else (sub[c] > v)           # NaN → False (not failed), as in GATK
        tot_fail |= fail.fillna(False)
        rows.append({"class": cl, "filter": name, "expression": e, "failing records": int(fail.sum()), "% of class": 100 * fail.mean() if len(sub) else np.nan})
    rows.append({"class": cl, "filter": "ANY", "expression": "", "failing records": int(tot_fail.sum()), "% of class": 100 * tot_fail.mean() if len(sub) else np.nan})
display(pd.DataFrame(rows).round(2))

## 8.3 Layer 1 — apply the site hard filters

If the plots suggest that the default thresholds are too strict or too lax for *your* data (they were tuned on human exomes/genomes), change `SNP_FILTERS` / `INDEL_FILTERS` in the parameter cell and re-run from here. A useful sanity check is the **Ti/Tv ratio** (transitions/transversions): for real SNPs it is ≈ 2–2.2 in mammals and often 1.2–1.8 in insects; for random errors it approaches 0.5. A filter that raises Ti/Tv of the retained set is removing mostly false positives.

In [ ]:
# 8.3  Layer 1: site hard filters → FILTER column
SITEF = D["vcf"] / "HypMis_final_sitefiltered.vcf.gz"
if not done(SITEF, str(SITEF) + ".tbi"):
    hard_filter(VCF_RAW, "final")
fc = filter_counts(SITEF)
fc["status"] = np.where(fc.FILTER == "PASS", "PASS", "filtered")
print(fc.groupby(["TYPE", "status"]).size().unstack(fill_value=0))
def titv(vcf, extra=""):
    line = run(f"bcftools stats {extra} {vcf} | grep '^TSTV'", quiet=True).stdout.split("\n")[0].split("\t")
    return float(line[4]) if len(line) > 4 else float("nan")
failed_vcf = WORK / "tmp" / "failed_sites.vcf.gz"
run(f"bcftools view -e 'FILTER=\"PASS\"' {SITEF} -Oz -o {failed_vcf}", quiet=True)
print(f"Ti/Tv  raw: {titv(VCF_RAW):.2f}  |  PASS records: {titv(SITEF, '-f PASS'):.2f}  |  records removed by the filters: {titv(failed_vcf):.2f}")
fl = fc[fc.status == "filtered"].FILTER.str.split(";").explode().value_counts()
print("\nrecords flagged by each filter (a record may fail several):"); print(fl.to_string())

## 8.4 Layer 2 — genotype filters → missing data

Site filters judge *the variant*; they say nothing about whether **an individual's genotype** is trustworthy. Two numbers per genotype matter:

* **DP** — number of reads supporting that genotype. With one or two reads a heterozygote cannot be distinguished from a homozygote.
* **GQ** — genotype quality (Phred): how much more likely the called genotype is than the second-best. `GQ ≥ 20` = 99% confidence.

Genotypes with `DP < MIN_GT_DP` or `GQ < MIN_GQ` are replaced by **`./.`**. This makes the file a proper *hard-call* VCF: every genotype that remains is reasonably certain; the uncertain ones are explicit missing data, which downstream tools (PLINK, vcftools) understand. *The price* is missingness; at low depth it can be large.

In [ ]:
# 8.4  Layer 2: genotype filters (DP/GQ) → ./.
GTF = D["vcf"] / "HypMis_final_genotypefiltered.vcf.gz"
if not done(GTF, str(GTF) + ".tbi"):
    t = WORK / "tmp"
    run(f"gatk {GATK_JAVA} VariantFiltration -R {REF} -V {SITEF} --genotype-filter-expression \"DP < {MIN_GT_DP} || GQ < {MIN_GQ}\" "
        f"--genotype-filter-name lowGT -O {t}/gtf1.vcf.gz", log=D["logs"] / "gt_filter.log", show=2, quiet=True)
    run(f"gatk {GATK_JAVA} SelectVariants -R {REF} -V {t}/gtf1.vcf.gz --exclude-filtered --set-filtered-gt-to-nocall -O {t}/gtf2.vcf.gz",
        log=D["logs"] / "gt_select.log", show=2, quiet=True)
    shutil.copy(t / "gtf2.vcf.gz", GTF); shutil.copy(t / "gtf2.vcf.gz.tbi", str(GTF) + ".tbi")
n_gtf = nvar(GTF)
miss = run(f"bcftools query -f '[%GT\\n]' {GTF} | awk '$1==\"./.\"||$1==\".|.\"||$1==\".\"{{m++}} END{{print m+0, NR}}'", quiet=True).stdout.split()
print(f"after site filters (PASS only, `--exclude-filtered`): {n_gtf:,} variants")
print(f"genotype entries set to missing (DP<{MIN_GT_DP} or GQ<{MIN_GQ}, plus originally missing): {int(miss[0]):,} of {int(miss[1]):,} = {100*int(miss[0])/max(int(miss[1]),1):.1f}%")

## 8.5 Layers 3 and 4 — depth, missingness, carriers, variant type

**Layer 3 — maximum depth.** Sites with total depth far above the average are usually collapsed repeats. We remove sites with `INFO/DP > mean + MAX_DP_SD × sd` (same idea as ANGSD's `-setMaxDepth`).

**Layer 4 — population-level filters.**
* `F_MISSING ≤ MAX_MISSING`: keep sites genotyped in enough individuals. Sites with large missingness give little information, and missingness correlated with a *phenotype* can create false associations in a GWAS.
* `N_PASS(GT="alt") ≥ MIN_CARRIERS`: after setting genotypes to missing, some sites may no longer have any individual carrying the alternative allele.
* `FINAL_SNPS_ONLY`: biallelic SNPs only (`-m2 -M2 -v snps`); most GWAS and population genetic tools expect this. Indels and multi-allelic sites stay in the “all variants” file.

In [ ]:
# 8.5  Layers 3 + 4 → final hard-call VCFs
dp = pd.Series(run(f"bcftools query -f '%INFO/DP\\n' {GTF}", quiet=True).stdout.split()).replace(".", np.nan).astype(float).dropna()
MAXDP = int(dp.mean() + MAX_DP_SD * dp.std()) if (MAX_DP_SD and len(dp) > 10) else None
print(f"site depth (INFO/DP): mean {dp.mean():.0f}, sd {dp.std():.0f} → max-depth cut-off = {MAXDP}")

ALL_VCF = D["vcf"] / "HypMis_GATK_hardcalls_allvariants.vcf.gz"
SNP_VCF = D["vcf"] / "HypMis_GATK_hardcalls.vcf.gz"
dp_expr = f"INFO/DP>{MAXDP}" if MAXDP else "0"
pop_expr = f'F_MISSING<={MAX_MISSING} && N_PASS(GT="alt")>={MIN_CARRIERS}'
run(f"bcftools view -e '{dp_expr}' {GTF} -Ou | bcftools view -i '{pop_expr}' -Oz -o {ALL_VCF} && bcftools index -t -f {ALL_VCF}", log=D["logs"] / "popfilter_all.log", show=2)
snp_sel = "-m2 -M2 -v snps" if FINAL_SNPS_ONLY else ""
run(f"bcftools view {snp_sel} {ALL_VCF} -Oz -o {SNP_VCF} && bcftools index -t -f {SNP_VCF}", log=D["logs"] / "popfilter_snp.log", show=2)

F3 = WORK / "tmp" / "f3_depth.vcf.gz"
run(f"bcftools view -e '{dp_expr}' {GTF} -Oz -o {F3}", quiet=True)
funnel = pd.DataFrame([
    ("raw joint call set (GenotypeGVCFs)", n_final_raw),
    ("after site hard filters (PASS)", n_gtf),
    ("after depth filter", nvar(F3)),
    ("after missingness / carriers (all variant types)", nvar(ALL_VCF)),
    ("final: biallelic SNPs" if FINAL_SNPS_ONLY else "final", nvar(SNP_VCF)),
], columns=["stage", "variants"])
funnel["removed"] = -funnel.variants.diff().fillna(0).astype(int)
display(funnel)
funnel.to_csv(D["tables"] / "filter_funnel.tsv", sep="\t", index=False)
print("\nFinal hard-call VCF :", SNP_VCF)
print("All variants (SNP+indel, hard calls):", ALL_VCF)

# Part 9 — Quality report of the hard-call VCF

Numbers worth looking at before using the VCF downstream:

* **Ti/Tv ratio** — see 8.3; a sudden drop signals false positives.
* **Per-individual missingness, heterozygosity and depth** — outliers indicate bad libraries, contamination (too many heterozygotes) or inbreeding.
* **Allele frequency spectrum** — in a population sample the number of variants falls steeply with allele frequency; GATK joint calling at low depth may lose singletons.
* **Variant density along the chromosome** — gaps or spikes = mappability problems or collapsed repeats.

In [ ]:
# 9.1  Per-sample statistics, AFS, Ti/Tv
st = run(f"bcftools stats -s - {SNP_VCF}", quiet=True).stdout.splitlines()
psc = pd.DataFrame([l.split("\t")[2:] for l in st if l.startswith("PSC")],
                   columns=["sample", "nRefHom", "nNonRefHom", "nHets", "nTs", "nTv", "nIndels", "avgDP", "nSingletons", "nHapRef", "nHapAlt", "nMissing"])
for c in psc.columns[1:]: psc[c] = psc[c].astype(float)
nsites = nvar(SNP_VCF)
psc["missing_%"] = 100 * psc.nMissing / max(nsites, 1)
psc["het_rate_%"] = 100 * psc.nHets / np.maximum(psc.nRefHom + psc.nNonRefHom + psc.nHets, 1)
psc = psc.merge(pheno[["sampleID", "morph"]], left_on="sample", right_on="sampleID", how="left")
print(f"final SNP set: {nsites:,} SNPs;  Ti/Tv = {titv(SNP_VCF):.2f}")

# allele-frequency spectrum from the hard calls
afs = []
p = subprocess.Popen(f"bcftools query -f '[%GT\\t]\\n' {SNP_VCF}", shell=True, stdout=subprocess.PIPE, text=True)
for line in p.stdout:
    g = [x for x in line.rstrip("\t\n").split("\t") if "." not in x]
    n = 2 * len(g); a = sum(x.count("1") for x in g)
    if n: afs.append(min(a, n - a) / n)
p.stdout.close(); p.wait()

fig, ax = plt.subplots(1, 4, figsize=(19, 3.8))
cols = psc.morph.map({"m": "#4c72b0", "i": "#dd8452"}).fillna("grey")
ax[0].bar(range(len(psc)), psc["missing_%"], color=cols); ax[0].set_title("missing genotypes per individual (%)"); ax[0].set_xticks(range(len(psc))); ax[0].set_xticklabels(psc["sample"], rotation=90, fontsize=6)
ax[1].bar(range(len(psc)), psc["het_rate_%"], color=cols); ax[1].set_title("heterozygosity (% of called SNPs)"); ax[1].set_xticks(range(len(psc))); ax[1].set_xticklabels(psc["sample"], rotation=90, fontsize=6)
ax[2].scatter(psc.avgDP, psc["missing_%"], c=cols); ax[2].set_xlabel("mean genotype depth at called sites"); ax[2].set_ylabel("missing %"); ax[2].set_title("depth vs missingness")
ax[3].hist(afs, bins=20, color="grey"); ax[3].set_xlabel("minor allele frequency (hard calls)"); ax[3].set_ylabel("SNPs"); ax[3].set_title("allele-frequency spectrum")
plt.tight_layout(); plt.savefig(D["figures"] / "final_vcf_quality.png", dpi=150); plt.show()
display(psc[["sample", "morph", "missing_%", "het_rate_%", "avgDP", "nSingletons"]].round(2).head(10))

In [ ]:
# 9.2  Variant density along the interval
pos = np.array(run(f"bcftools query -f '%POS\\n' {SNP_VCF}", quiet=True).stdout.split(), dtype=np.int64)
s_, e_ = (REGION if (REGION and len(CONTIGS) == 1) else (1, flen[CONTIGS[0]]))
bins = np.arange(s_, e_ + 100_001, 100_000)
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.hist(pos, bins=bins, color="#55A868"); ax.set_xlabel(f"{CONTIGS[0]} position (bp)"); ax.set_ylabel("SNPs per 100 kb")
ax.set_title("Density of the final SNPs (gaps = low mappability / low coverage)")
plt.tight_layout(); plt.savefig(D["figures"] / "snp_density.png", dpi=150); plt.show()
print(f"{len(pos):,} SNPs in {(e_-s_)/1e6:.1f} Mb → 1 SNP per {(e_-s_)/max(len(pos),1):.0f} bp")

## 9.3 Comparison with the ANGSD call set (optional)

If you ran the previous notebook (`association_HypMis/vcf/HypMis_ANGSD_calls.vcf.gz`) we can compare **which positions** each method reports. They answer different questions: ANGSD calls SNPs from *genotype likelihoods* with an allele-frequency test (MAF ≥ 5%), GATK from per-individual *haplotype-based* calls with hard filters; so we expect GATK to report more rare variants and ANGSD to be more permissive at low depth.

In [ ]:
# 9.3  Positions shared with the ANGSD call set (if available)
ang = PROJECT_DIR / "association_HypMis" / "vcf" / "HypMis_ANGSD_calls.vcf.gz"
if ang.exists():
    c = CONTIGS[0]
    A = set(map(int, run(f"bcftools query -r {INTERVALS[0]} -f '%POS\\n' {ang}", quiet=True).stdout.split()))
    G = set(map(int, pos))
    print(f"in {INTERVALS[0]}:  ANGSD {len(A):,} sites | GATK hard-call SNPs {len(G):,} | shared {len(A & G):,}")
    print(f"  GATK sites also found by ANGSD: {100*len(A & G)/max(len(G),1):.1f}%   |   ANGSD sites also found by GATK: {100*len(A & G)/max(len(A),1):.1f}%")
else:
    print("ANGSD call set not found – skipping the comparison.")

In [ ]:
# 9.4  Summary
summary = {
    "contigs": CONTIGS, "intervals": INTERVALS, "n_samples": N,
    "markdup": RUN_MARKDUP, "indel_realignment": bool(REALIGNED), "bqsr_applied": bool(BQSR_APPLIED),
    "heterozygosity_prior": HET_PRIOR, "site_filters_SNP": SNP_FILTERS, "site_filters_indel": INDEL_FILTERS,
    "genotype_filters": {"min_DP": MIN_GT_DP, "min_GQ": MIN_GQ}, "max_depth_cutoff": MAXDP, "max_missing": MAX_MISSING,
    "variants_raw": int(n_final_raw), "variants_final_snps": int(nsites), "titv_final": round(titv(SNP_VCF), 3),
    "final_vcf": str(SNP_VCF),
}
(D["tables"] / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("\nEverything is saved in", OUT)

# Part 10 — Discussion, limitations, going further

## 10.1 Take-home messages

1. **Every step has a reason**: duplicates → inflated evidence; (mis)alignment around indels → false SNPs; base qualities → miscalibrated likelihoods; low depth → uncertain genotypes; paralogs → pile-ups of reads.
2. **GATK4 integrates realignment** into HaplotypeCaller; the separate `IndelRealigner` step belongs to GATK3-era pipelines (and to callers that do not reassemble).
3. **Non-model organisms lack truth sets**: no VQSR, no known-site BQSR. The workarounds are *bootstrapping* and *hard filters*, which require you to **inspect the distributions** and adapt thresholds.
4. **GVCF + joint calling** makes cohorts extensible and calls more accurate than single-sample calling.
5. **Hard calls hide uncertainty.** The more missing/shallow the data, the more the statistics (allele frequency, heterozygosity, association tests) depend on how the filters were chosen. Record **all** parameters (the notebook writes `summary.json`).

## 10.2 Scaling up to the whole genome and all 330 individuals

| What | How |
|---|---|
| Whole genome | run per **contig** (or per 10–20 Mb interval) in parallel: `HaplotypeCaller -L contig` and `GenomicsDBImport -L contig`; concatenate with `bcftools concat`. A cluster (SLURM) job array or a cloud batch system is the usual approach. |
| 330 samples | GVCFs per sample once; `GenomicsDBImport --batch-size 50` and `--max-num-intervals-to-import-in-parallel`; the joint-genotyping time grows roughly linearly with the number of samples. |
| Sex chromosomes | Z: females are hemizygous → call with `-ploidy 1` for females or analyse males only; W: haploid, females only. |
| Memory | HaplotypeCaller ~4–8 GB per process; GenotypeGVCFs 8–16 GB for hundreds of samples. |
| Complete BAMs | the extraction step (Part 2.5) is for the practical; on a cluster use full BAMs so that duplicates of pairs spanning contigs are marked correctly. |

## 10.3 Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `Sequence dictionary mismatch` | the BAM was aligned to a different FASTA than the one given to GATK | use the identical file; the check in 2.3 stops you early |
| `SAM/BAM file ... has no read groups` | read group missing | the notebook rewrites them (`NEED_RG_FIX`) |
| `GenomicsDBImport` fails/hangs on Drive | a lot of small writes on Drive | we use local disk; check free space (`!df -h /content`) |
| `java.lang.OutOfMemoryError` | heap too small for the number of parallel jobs | set `N_PAR = 1` and/or `MEM_GB = 6` |
| Almost no variants survive | depth too low for DP/GQ filters | reduce `MIN_GT_DP`, `MIN_GQ`, raise `MAX_MISSING`; look at 2.6 |
| Very high missingness in some individuals | low depth or poor library | exclude those individuals or re-sequence |
| `IndelRealigner` not found | GATK4 has no realigner | use the GATK3.8 environment (1.3) or skip |
| Colab disconnects | session limit | re-run from the top; GVCFs and VCFs on Drive are re-used |

## 10.4 Suggested exercises

1. Set `HET_PRIOR` to 0.001 (GATK default) and compare the number of raw variants and Ti/Tv.
2. Tighten and relax `MIN_GQ` (10, 30, 50): how do missingness, het-rate and Ti/Tv change?
3. Set `BQSR_ROUNDS = 0`: does the call set change? Look at both call sets with `bcftools isec`.
4. Use `RUN_INDEL_REALIGNMENT = False`: compare the number of indels and SNPs within 5 bp of indels.
5. Change `MAX_DP_SD` to 2 or `None`. Where on the chromosome do the removed sites cluster?
6. Compare the GATK call set with the ANGSD one at the M locus (positions 6.73–6.74 Mb on chromosome 26).
7. Which individuals have unusually high heterozygosity or missingness? What would you do?

## 10.5 References

* McKenna A et al. (2010) The Genome Analysis Toolkit. *Genome Research* 20:1297.
* DePristo MA et al. (2011) A framework for variation discovery and genotyping using next-generation DNA sequencing data. *Nature Genetics* 43:491.
* Van der Auwera GA & O'Connor BD (2020) *Genomics in the Cloud*. O'Reilly (GATK Best Practices).
* Poplin R et al. (2018) Scaling accurate genetic variant discovery to tens of thousands of samples. *bioRxiv* 201178 (GVCF + joint calling).
* GATK documentation: <https://gatk.broadinstitute.org/>.